# Penalty contact in LowLevelFEM

This short example demonstrates the basic penalty contact formulation in
LowLevelFEM.

Two elastic bodies are initially tangent. Contact is generated by prescribing
a small downward displacement of the upper body in three equal load steps.

For frictionless normal contact,

$$
K_c = \int_{\Gamma_c} G^T c_n G\,d\Gamma,
$$

where `G = ContactGap(C)` is the contact-gap operator and `c_n` is the penalty
stiffness.

At every iteration the contact residual is

$$
r_c = K_c(r+u),
$$

and the displacement correction is obtained from

$$
(K+K_c)\Delta u = -R.
$$


In [8]:
using LowLevelFEM, LinearAlgebra

openGeometry("2D-Hertz.geo")


## Model


In [9]:
matU = Material("upper", E=2e5, ν=0.3)
matL = Material("lower", E=2e5, ν=0.3)

U = Field(
    [matU, matL],
    type=:VectorField,
    dim=2,
    fieldName=:u
)

ε = SymGrad
K =
    ∫(ε(U) ⋅ D(:PlaneStrain, matL) ⋅ ε(U), Ω="lower") +
    ∫(ε(U) ⋅ D(:PlaneStrain, matU) ⋅ ε(U), Ω="upper")

# Zero external load.
f = ∫(U ⋅ [0, 0], Γ="top");


## Initial state and contact pair

The bodies are initially tangent. The master normal points downward, therefore
`normal_sign=-1`.

The sign convention is

- `gap < 0`: penetration,
- `gap > 0`: open contact.


In [10]:
support0 = [
    BoundaryCondition("top",    ux=0, uy=0),
    BoundaryCondition("bottom", uy=0),
    BoundaryCondition("left",   ux=0),
    BoundaryCondition("right",  ux=0)
]

u0 = solveField(K, f, support=support0)

C = contact(
    U,
    master="master",
    slave="slave",
    displacement=u0,
    normal_sign=-1
)

G = ContactGap(C)
r = nodePositionVector(U)
r = projectTo2D(r)

cn = 1e5


100000.0

## Boundary conditions for the incremental solution

Each load step first applies a prescribed downward displacement increment to
the upper body. The nonlinear contact corrections then use homogeneous
boundary conditions, so the prescribed displacement is preserved.


In [11]:
support_increment = [
    BoundaryCondition("top",    ux=0, uy=0),
    BoundaryCondition("bottom", uy=0),
    BoundaryCondition("left",   ux=0),
    BoundaryCondition("right",  ux=0)
]

function loadIncrementSupport(Δindent)
    [
        BoundaryCondition("top",    ux=0, uy=-Δindent),
        BoundaryCondition("bottom", uy=0),
        BoundaryCondition("left",   ux=0),
        BoundaryCondition("right",  ux=0)
    ]
end

free = freeDoFs(U, support_increment)

freeNorm(v::VectorField) =
    LinearAlgebra.norm(elementsToNodes(v).a[free, 1])


freeNorm (generic function with 1 method)

## Three displacement steps

The total indentation is divided into three equal increments. Within each
step, a simple relaxed fixed-point/Newton-like contact iteration is used.


In [12]:
indentation_final = 0.40
nsteps = 3
Δindent = indentation_final / nsteps

maxiter = 25
tol = 1e-3
ω = 0.5

u = copy(u0)

for step in 1:nsteps

    println()
    println("Load step ", step, "/", nsteps)

    # Elastic predictor for the new prescribed displacement level.
    Δu_load = solveField(Symmetric(K), f, support=loadIncrementSupport(Δindent))

    u_it = u + Δu_load

    for iter in 1:maxiter

        # Penalty contact tangent on the current configuration.
        Kc = ∫(G ⋅ cn ⋅ G; updateFrom=u_it)

        # Total residual.
        rc = Kc * (r + u_it)
        R = K * u_it - f + rc

        # Contact correction with homogeneous essential BCs.
        Δu = solveField(Symmetric(K + Kc), -R, support=support_increment)

        err = freeNorm(Δu) / max(freeNorm(u_it), eps(Float64))

        println("  iter = ", iter, ", error = ", err)

        # Relaxed update.
        u_it = u_it + ω * Δu

        err < tol && break
    end

    u = u_it
end



Load step 1/3
  iter = 1, error = 0.24833175343053562
  iter = 2, error = 0.14613813636796572
  iter = 3, error = 0.08789609990400728
  iter = 4, error = 0.05598916870107143
  iter = 5, error = 0.033945279591775505
  iter = 6, error = 0.02064602395786612
  iter = 7, error = 0.011392340538630537
  iter = 8, error = 0.005711675383706264
  iter = 9, error = 0.003163465882292529
  iter = 10, error = 0.0015831734543158998
  iter = 11, error = 0.0007920316686554653

Load step 2/3
  iter = 1, error = 0.17532336526196995
  iter = 2, error = 0.09919171906875421
  iter = 3, error = 0.05681442930181397
  iter = 4, error = 0.03379412170189818
  iter = 5, error = 0.01982474684796446
  iter = 6, error = 0.011247490900625915
  iter = 7, error = 0.005992050693436114
  iter = 8, error = 0.0033328844650220182
  iter = 9, error = 0.0017448230549568977
  iter = 10, error = 0.0008837576049567847

Load step 3/3
  iter = 1, error = 0.13551593021189529
  iter = 2, error = 0.07438736052171545
  iter = 3, erro

## Results

The full nodal contact state is updated once after the solution.

The penalty pressure is

$$
p_n = \max(-c_n g_n,0).
$$


In [13]:
ε = [∂x(u[1]), ∂y(u[2]), ∂y(u[1])+∂x(u[2])]
σ = D(:PlaneStrain, matL) * [ε[1], ε[2], ε[3]]

showElementResults(σ[1], name="σx")
showElementResults(σ[2], name="σy")
showElementResults(σ[3], name="τxy")
showDoFResults(√((σ[1]^2 + σ[2]^2 + (σ[1]-σ[2])^2 + 3*σ[3]^2)/ 2), name="σ von Mises")

3

In [14]:
updateContact!(C, u)

gap = ContactGap(C, u; gauss=4)

pressure = mapScalarField(
    g -> max(-cn * g, 0.0),
    gap
)

showDoFResults(
    u,
    name="displacement",
    factor=1,
    visible=false
)

plotOnBeam(
    "slave",
    nodesToElements(gap),
    name="gap"
)

plotOnBeam(
    "slave",
    nodesToElements(pressure),
    name="contact pressure"
)

plotOnBeam(
    "slave",
    nodesToElements(-σ[2]),
    name="contact stress"
)

openPostProcessor()


## Notes

This example intentionally keeps the nonlinear algorithm simple.

For more difficult contact problems, useful extensions include adaptive load
stepping, line search, augmented Lagrangian methods and mortar formulations.
The penalty stiffness should also be chosen with care: a very large value
reduces penetration but can deteriorate conditioning and introduce
mesh-scale pressure oscillations.
